In [152]:
!pip install pyspark

In [153]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

# Q1

In [154]:
!wget -q https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt

In [155]:
rdd = sc.textFile("sherlock.txt")

In [156]:
rdd.take(5)

['A STUDY IN SCARLET.', '', '', '', '']

In [157]:
words = rdd.flatMap(lambda txt: txt.split(' '))
words.take(5)

['A', 'STUDY', 'IN', 'SCARLET.', '']

In [158]:
filtered = words.filter(lambda w: len(w) >= 2)  # Example에서 IN도 포함하므로 2글자인 것도 포함
filtered.take(5)

['STUDY', 'IN', 'SCARLET.', 'PART', 'I.']

In [159]:
heads = filtered.map(lambda w: (w, w.lower()[:2]))
heads.take(5)

[('STUDY', 'st'),
 ('IN', 'in'),
 ('SCARLET.', 'sc'),
 ('PART', 'pa'),
 ('I.', 'i.')]

In [160]:
heads_count = filtered.map(lambda w: (w.lower()[:2], 1))
heads_count.take(5)

[('st', 1), ('in', 1), ('sc', 1), ('pa', 1), ('i.', 1)]

In [161]:
counts = heads_count.reduceByKey(lambda x, y: x + y)
counts.take(5)

[('i.', 9), ('(b', 1), ('of', 1256), ('h.', 2), ('m.', 2)]

In [162]:
under32 = counts.filter(lambda x: x[1] < 32)
under32.take(5)

[('i.', 9), ('(b', 1), ('h.', 2), ('m.', 2), ('18', 2)]

In [163]:
under32.map(lambda x: x[1]).reduce(lambda x, y: x + y)

830

In [164]:
# sol
rdd.flatMap(lambda txt: txt.split(' ')) \
    .filter(lambda w: len(w) >= 2) \
    .map(lambda w: (w.lower()[:2], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .filter(lambda x: x[1] < 32) \
    .map(lambda x: x[1]) \
    .reduce(lambda x, y: x + y)

830

# Q2

In [165]:
!wget -q -O int-pair-input.txt https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt

rdd = sc.textFile("int-pair-input.txt")
rdd.take(5)

['(2, 96)', '(19, 78)', '(13, 83)', '(17, 78)', '(32, 86)']

In [166]:
# sol
pairs = rdd.map(lambda line: line.replace("(", "").replace(")", "").split(",")) \
           .map(lambda x: (int(x[0]), int(x[1])))

# 첫 번째 정수 쌍의 첫 번째 원소
a = pairs.first()[0]

# 모든 정수 쌍의 첫 번째 원소 중 최댓값
b = pairs.map(lambda x: x[0]) \
         .reduce(lambda x, y: x if x > y else y)

# 모든 정수 쌍의 첫 번째 원소 중 최솟값
c = pairs.map(lambda x: x[0]) \
         .reduce(lambda x, y: x if x < y else y)

# 첫 번째 원소가 1인 정수 쌍의 개수
d = pairs.filter(lambda x: x[0] == 1).count()

# 첫 번째 원소가 2인 정수 쌍의 개수
e = pairs.filter(lambda x: x[0] == 2).count()

print(a, b, c, d, e)
print(a + b + c + d + e)

2 32 0 31 28
93


# Q3

In [167]:
!git clone https://github.com/nongaussian/class-2024-datamining.git

fatal: destination path 'class-2024-datamining' already exists and is not an empty directory.


In [168]:
path = "/content/class-2024-datamining/data/World Hapiness Report/"

rdd15 = sc.textFile(path + "2015.csv")
rdd16 = sc.textFile(path + "2016.csv")
rdd17 = sc.textFile(path + "2017.csv")
rdd18 = sc.textFile(path + "2018.csv")
rdd19 = sc.textFile(path + "2019.csv")

In [169]:
print(rdd15.take(5))
print(rdd16.take(5))
print(rdd17.take(5))
print(rdd18.take(5))
print(rdd19.take(5))

['Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)', 'Switzerland,1,7.587,1.39651,1.34951,0.94143,0.66557,0.41978', 'Iceland,2,7.561,1.30232,1.40223,0.94784,0.62877,0.14145', 'Denmark,3,7.527,1.32548,1.36058,0.87464,0.64938,0.48357', 'Norway,4,7.522,1.459,1.33095,0.88521,0.66973,0.36503']
['Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)', 'Denmark,1,7.526,1.44178,1.16374,0.79504,0.57941,0.44453', 'Switzerland,2,7.509,1.52733,1.14524,0.86303,0.58557,0.41203', 'Iceland,3,7.501,1.42666,1.18326,0.86733,0.56624,0.14975', 'Norway,4,7.498,1.57744,1.1269,0.79579,0.59609,0.35776']
['Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)', 'Norway,1,7.537000179,1.616463184,1.53352356,0.796666503,0.635422587,0.362012237,0.315963835', 'Denm

In [170]:
# sol1
# Country로 시작하지 않는 행 = 데이터만 담고 있는 것
r15 = rdd15.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))  # 국가명과 행복지수만 가져옴

r16 = rdd16.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r17 = rdd17.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r18 = rdd18.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

r19 = rdd19.filter(lambda x: not x.startswith("Country")) \
           .map(lambda x: x.split(",")) \
           .map(lambda x: (x[0], float(x[2])))

In [171]:
# 확인: key인 국가명으로 rdd 결합
check = r15.join(r16)
print(check.take(5))  # x[0]: 나라, x[1]: 행복지수

[('Sweden', (7.364, 7.291)), ('Australia', (7.284, 7.313)), ('Israel', (7.278, 7.267)), ('Mexico', (7.187, 6.778)), ('United Arab Emirates', (6.901, 6.573))]


In [172]:
# 확인: 나라별 행복지수 감소/증가
check = r15.join(r16)

print("2015 → 2016 감소한 국가:")
print(check.filter(lambda x: x[1][0] > x[1][1]).take(5))

print("2015 → 2016 증가 또는 동일한 국가:")
print(check.filter(lambda x: x[1][0] <= x[1][1]).take(5))

2015 → 2016 감소한 국가:
[('Sweden', (7.364, 7.291)), ('Israel', (7.278, 7.267)), ('Mexico', (7.187, 6.778)), ('United Arab Emirates', (6.901, 6.573)), ('Venezuela', (6.81, 6.084))]
2015 → 2016 증가 또는 동일한 국가:
[('Australia', (7.284, 7.313)), ('Trinidad and Tobago', (6.168, 6.168)), ('Italy', (5.948, 5.977)), ('Mauritius', (5.477, 5.648)), ('Romania', (5.124, 5.528))]


In [173]:
# sol2
result = r15.join(r16) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: (x[0], x[1][1])) \
    .join(r17) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: (x[0], x[1][1])) \
    .join(r18) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: (x[0], x[1][1])) \
    .join(r19) \
    .filter(lambda x: x[1][0] <= x[1][1]) \
    .map(lambda x: x[0])

result.collect()

['Gabon',
 'Malta',
 'Senegal',
 'Chad',
 'Czech Republic',
 'Cambodia',
 'Tajikistan',
 'Congo (Brazzaville)',
 'Latvia',
 'Finland',
 'Dominican Republic',
 'Honduras',
 'Syria',
 'Estonia',
 'Mongolia',
 'Poland',
 'Togo',
 'Slovakia',
 'Hungary',
 'Cameroon',
 'Romania',
 'Benin',
 'Burkina Faso',
 'Serbia',
 'Ivory Coast',
 'Philippines',
 'Portugal',
 'Niger']

In [174]:
# 확인: Gabon 실제 행복지수
print(r15.filter(lambda x: x[0] == "Gabon").collect())
print(r16.filter(lambda x: x[0] == "Gabon").collect())
print(r17.filter(lambda x: x[0] == "Gabon").collect())
print(r18.filter(lambda x: x[0] == "Gabon").collect())
print(r19.filter(lambda x: x[0] == "Gabon").collect())

[('Gabon', 3.896)]
[('Gabon', 4.121)]
[('Gabon', 4.465000153)]
[('Gabon', 4.758)]
[('Gabon', 4.799)]
